# lotto-lab 歷史策略走步外驗

這是 `research_backtest.py` 的可重現分析伴隨筆記本。它只讀取研究輸出，不會寫入 `records/` 或改動 v1 凍結票。


## tl;dr

- 經濟決策以「不參與」為基準，因為合法號碼組合等機率且每注有正成本。
- 條件式模擬政策必須在封存測試集相對 `random_5` 的 95% 區間完全高於 0，並通過 replica 與逐週一致性門檻。
- 威力彩的 validation 冠軍在 holdout 相對隨機為 +0.66%，95% 區間 [-2.95%, +4.31%]；優勢未證明。
- 大樂透的 validation 冠軍在 holdout 相對隨機為 -2.97%，95% 區間 [-9.80%, +3.81%]；優勢未證明，因此條件式政策維持 `random_5`。


## Context & Methods

### Key Assumptions

- 每週選號只能使用該週週一以前的歷史。
- 前 50% 週用於訓練與兩輪參數搜尋，接續 25% 用於政策選擇，最後 25% 為一次性封存測試集。
- 主要指標排除頭獎與貳獎，再和同週、同 replica 的五注純隨機政策配對比較，避免單一極端獎金主導選擇。
- 同一週的五注套用該週所有實際開獎；春節加開期也按實際成本計入。


In [1]:
# 1. 載入研究輸出（僅 Python 標準函式庫）
from __future__ import annotations

import csv
import json
from collections import defaultdict
from pathlib import Path

root = Path.cwd()
if not (root / "research" / "results" / "strategy_research.json").exists():
    candidates = [root, *root.parents]
    root = next(
        path for path in candidates
        if (path / "research" / "results" / "strategy_research.json").exists()
    )
results_dir = root / "research" / "results"
study = json.loads((results_dir / "strategy_research.json").read_text(encoding="utf-8"))
with open(results_dir / "strategy_summary.csv", encoding="utf-8-sig", newline="") as handle:
    summaries = list(csv.DictReader(handle))
with open(results_dir / "policy_weekly.csv", encoding="utf-8-sig", newline="") as handle:
    weekly = list(csv.DictReader(handle))
{
    "research_id": study["research_id"],
    "generated_at": study["generated_at"],
    "replicates": study["replicates"],
    "summary_rows": len(summaries),
    "weekly_rows": len(weekly),
}


{'research_id': 'walkforward-v1',
 'generated_at': '2026-07-18T17:04:23+08:00',
 'replicates': 8,
 'summary_rows': 420,
 'weekly_rows': 9925}

## Data

資料來自專案內保存的台彩官方 API 月快取。以下檢查涵蓋期號唯一性、號碼合法性、獎金欄位完整性與每週實際開獎數。


In [2]:
# 2. 顯示資料品質摘要
data_quality = [
    {
        "遊戲": row["game_name"],
        "期數": row["draws"],
        "週數": row["weeks"],
        "日期範圍": f'{row["date_min"]} → {row["date_max"]}',
        "非一般排程期": row["off_regular_schedule_draws"],
        "品質": row["quality_status"],
    }
    for row in study["data_quality"]
]
data_quality


[{'遊戲': '威力彩',
  '期數': 1929,
  '週數': 965,
  '日期範圍': '2008-01-24 → 2026-07-16',
  '非一般排程期': 0,
  '品質': 'pass'},
 {'遊戲': '大樂透',
  '期數': 2153,
  '週數': 1020,
  '日期範圍': '2007-01-02 → 2026-07-17',
  '非一般排程期': 113,
  '品質': 'pass'}]

## Results

先在訓練集完成粗搜尋與局部細調，再用驗證集從五種投資組合選一次。下表只顯示選定政策在封存測試集的結果。


In [3]:
# 3. 封存測試集決勝
holdout_decisions = []
for game, result in study["selected"].items():
    row = result["holdout"]
    holdout_decisions.append({
        "遊戲": row["game_name"],
        "驗證集選定政策": result["policy_id"],
        "原始 ROI": f'{row["raw_roi_mean"]:+.2%}',
        "相對隨機穩健 ROI 差": f'{row["delta_robust_roi_mean"]:+.2%}',
        "95% 區間": f'[{row["delta_ci_low"]:+.2%}, {row["delta_ci_high"]:+.2%}]',
        "外驗優勢": "通過" if result["edge_proven"] else "未證明",
        "條件式決策": result["conditional_decision"],
    })
holdout_decisions


[{'遊戲': '威力彩',
  '驗證集選定政策': 'current_ensemble',
  '原始 ROI': '-77.94%',
  '相對隨機穩健 ROI 差': '+0.66%',
  '95% 區間': '[-2.95%, +4.31%]',
  '外驗優勢': '未證明',
  '條件式決策': 'random_5'},
 {'遊戲': '大樂透',
  '驗證集選定政策': 'current_ensemble',
  '原始 ROI': '-68.72%',
  '相對隨機穩健 ROI 差': '-2.97%',
  '95% 區間': '[-9.80%, +3.81%]',
  '外驗優勢': '未證明',
  '條件式決策': 'random_5'}]

In [4]:
# 4. 比較各最終政策在 validation 與 holdout 的相對隨機差異
comparison = []
for row in summaries:
    if row["stage"] != "final_policy" or row["split"] == "train":
        continue
    comparison.append({
        "遊戲": row["game_name"],
        "資料段": row["split"],
        "政策": row["policy_id"],
        "相對隨機穩健 ROI 差": f'{float(row["delta_robust_roi_mean"]):+.2%}',
        "原始 ROI": f'{float(row["raw_roi_mean"]):+.2%}',
    })
comparison


[{'遊戲': '威力彩',
  '資料段': 'holdout',
  '政策': 'current_ensemble',
  '相對隨機穩健 ROI 差': '+0.66%',
  '原始 ROI': '-77.94%'},
 {'遊戲': '威力彩',
  '資料段': 'validation',
  '政策': 'current_ensemble',
  '相對隨機穩健 ROI 差': '+2.04%',
  '原始 ROI': '-79.12%'},
 {'遊戲': '威力彩',
  '資料段': 'holdout',
  '政策': 'random_5',
  '相對隨機穩健 ROI 差': '+0.00%',
  '原始 ROI': '-78.60%'},
 {'遊戲': '威力彩',
  '資料段': 'validation',
  '政策': 'random_5',
  '相對隨機穩健 ROI 差': '+0.00%',
  '原始 ROI': '-81.17%'},
 {'遊戲': '威力彩',
  '資料段': 'holdout',
  '政策': 'trained_best_5',
  '相對隨機穩健 ROI 差': '-1.68%',
  '原始 ROI': '-80.28%'},
 {'遊戲': '威力彩',
  '資料段': 'validation',
  '政策': 'trained_best_5',
  '相對隨機穩健 ROI 差': '+0.51%',
  '原始 ROI': '-80.66%'},
 {'遊戲': '威力彩',
  '資料段': 'holdout',
  '政策': 'trained_blend',
  '相對隨機穩健 ROI 差': '-0.25%',
  '原始 ROI': '-78.85%'},
 {'遊戲': '威力彩',
  '資料段': 'validation',
  '政策': 'trained_blend',
  '相對隨機穩健 ROI 差': '+1.38%',
  '原始 ROI': '-79.78%'},
 {'遊戲': '威力彩',
  '資料段': 'holdout',
  '政策': 'trained_family_ensemble',
  '相對隨機穩健 ROI 差': '-0.72

In [5]:
# 5. 選定政策與 random_5 的 holdout 累計損益終值（跨 replica 平均）
curves = defaultdict(float)
for row in weekly:
    if row["split"] != "holdout":
        continue
    selected_policy = study["selected"][row["game"]]["policy_id"]
    if row["policy_id"] not in {"random_5", selected_policy}:
        continue
    curves[(row["game_name"], row["policy_id"])] += float(row["raw_pnl_mean"])
[
    {"遊戲": game, "政策": policy, "holdout 累計損益": round(value)}
    for (game, policy), value in sorted(curves.items())
]


[{'遊戲': '大樂透', '政策': 'current_ensemble', 'holdout 累計損益': -98444},
 {'遊戲': '大樂透', '政策': 'random_5', 'holdout 累計損益': -94195},
 {'遊戲': '威力彩', '政策': 'current_ensemble', 'holdout 累計損益': -188612},
 {'遊戲': '威力彩', '政策': 'random_5', 'holdout 累計損益': -190200}]

## Takeaways

- 「歷史回跑第一名」不等於可使用的優勢；只有封存測試集門檻決定是否更換條件式政策。
- 兩款遊戲的封存測試區間都跨過 0，研究結論回到 `random_5`；不得再用這份 holdout 反覆調參。
- 經濟層的最佳決策獨立於條件式選號研究：在負期望值與正成本下維持不參與。
